# Belgian convective test events (D3)

D3 (#27): the final verification uses recent Belgian convective events covered by CIRRUS, chosen before any
training. This notebook prepares the choice; **it does not choose**. The criteria are to be agreed with Vito
(review of #27), from the literature collected in `docs/method.md`, *Test events (D3)*.

1. **Daily rain maxima on CIRRUS:** for every day of 7/2024–11/2026 and every pixel of Belgium, the highest
   1 h sum and the 24 h sum.
2. **Days ranked by heavy-rain objects:** contiguous areas above 25 mm in 1 h, at least 9 km² (CatRaRE).
3. **Check against the events in the reports** of `docs/method.md`, *Reported events*.
4. **Maps** of the highest-ranked days.

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import glob
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage

from opera_ds.odim_io import read_odim_composite

OUT = RESULTS_DIR / "d3_events"
OUT.mkdir(parents=True, exist_ok=True)

## Settings

- **Data:** CIRRUS `DBZH` crops at 1 km and 5 min (`docs/data.md`, *Derived datasets*), 7/2024–9/11/2026.
  CIRRUS is missing on 7–17/10/2024 and 13–15/5/2025.
- **Belgium:** the Natural Earth mask `data/mask_BEL_1km.npy` (30,667 pixels).
- **Rain rate:** Marshall–Palmer (Z = 200 R^1.6), from 7 dBZ and capped at 55 dBZ (Imhoff et al. 2020); values
  above 70 dBZ count as no rain (H2, impossible values). A missing 5 min step counts as no rain.
- **Sums:** rolling 1 h sums of 12 steps inside each UTC day, and the 24 h sum of the day.
- **Thresholds in the literature:** 20, 31 and above 50 mm in 1 h (RMI warnings, yellow, orange, red); 25 mm in
  1 h with objects of at least 9 km² (CatRaRE, Lengfeld et al. 2021).
- **Caveat:** the composite is a column maximum, so these sums are an **index**, higher than the rain at the
  ground; fixed clutter not yet cleaned (H2) can also reach 25 mm in 1 h on many days at the same pixels.

In [ ]:
MASK = np.load(DATA_DIR / "mask_BEL_1km.npy")
rows_, cols_ = np.where(MASK.any(1))[0], np.where(MASK.any(0))[0]
R0, R1, C0, C1 = int(rows_[0]), int(rows_[-1]) + 1, int(cols_[0]), int(cols_[-1]) + 1
BEL = MASK[R0:R1, C0:C1]


def rate(path):
    f = read_odim_composite(path, "DBZH")
    r0, c0 = int(f.where.get("crop_row0", 0)), int(f.where.get("crop_col0", 0))
    sl = (slice(R0 - r0, R1 - r0), slice(C0 - c0, C1 - c0))
    z = f.values[sl]
    ok = f.coverage[sl] & ~f.undetect[sl] & (z <= 70)
    zc = np.clip(z, None, 55.0)
    return np.where(ok & (zc >= 7), (10 ** (zc / 10) / 200.0) ** (1 / 1.6), 0.0).astype(np.float32), f.time


def day_maps(files):
    acc = np.zeros((288,) + BEL.shape, np.float32)
    for p in files:
        try:
            r, t = rate(p)
        except Exception:      # unreadable file: counted as missing
            continue
        acc[(t.hour * 60 + t.minute) // 5] = r * (5 / 60)
    cs = np.cumsum(acc, 0)
    h1 = cs[11:] - np.concatenate([np.zeros((1,) + BEL.shape, np.float32), cs[:-12]])
    return np.where(BEL, h1.max(0), 0).astype(np.float16), cs[-1].astype(np.float16)


def year_maps(year):
    path = OUT / f"cirrus_bel_dailymaps_{year}.npz"
    if path.exists():
        z = np.load(path)
        return list(z["days"]), z["h1max"].astype(np.float32), z["d24"].astype(np.float32)
    days = defaultdict(list)
    for p in sorted(glob.glob(str(DATA_DIR / "cirrus_bel" / str(year) / "*" / "*.h5"))):
        days[os.path.basename(p).split("@")[1][:8]].append(p)
    keys = sorted(days)
    maps = [day_maps(days[k]) for k in keys]
    np.savez_compressed(path, days=np.array(keys), h1max=np.stack([m[0] for m in maps]), d24=np.stack([m[1] for m in maps]),
                        bbox=np.array([R0, R1, C0, C1]), mask=BEL)
    return keys, np.stack([m[0] for m in maps]).astype(np.float32), np.stack([m[1] for m in maps]).astype(np.float32)


DAYS, H1, D24 = [], [], []
for year in (2024, 2025, 2026):
    k, h, d = year_maps(year)
    DAYS += k; H1.append(h); D24.append(d)
H1, D24 = np.concatenate(H1), np.concatenate(D24)
print(len(DAYS), "days,", DAYS[0], "to", DAYS[-1])

## Days ranked by heavy-rain objects

For each day: the objects of contiguous pixels whose highest 1 h sum is at least 25 mm (4-connected), kept when
they have at least 9 km² (CatRaRE); the area of all such objects is the ranking key. Fixed clutter shows as
pixels above 25 mm on many days: their share in each day's objects is printed, not removed (H2 is not merged
yet).

In [ ]:
persistent = (H1 >= 25).sum(0) >= 0.05 * len(DAYS)
rows = []
for i, d in enumerate(DAYS):
    b = (H1[i] >= 25) & BEL
    lab, n = ndimage.label(b)
    if n == 0:
        rows.append((d, 0, 0, 0.0, float(H1[i].max()), int((H1[i] >= 20).sum()), int((H1[i] >= 31).sum()), int((H1[i] > 50).sum())))
        continue
    size = ndimage.sum(b, lab, range(1, n + 1))
    big = np.where(size >= 9)[0] + 1
    inb = np.isin(lab, big)
    rows.append((d, len(big), int(inb.sum()), float(persistent[inb].mean()) if inb.any() else 0.0, float(H1[i].max()),
                 int((H1[i] >= 20).sum()), int((H1[i] >= 31).sum()), int((H1[i] > 50).sum())))
rank = pd.DataFrame(rows, columns=["day", "objects", "area_km2", "share_persistent", "max_1h_mm", "px_1h_20", "px_1h_31", "px_1h_50"])
rank["rank"] = rank.area_km2.rank(ascending=False, method="min").astype(int)
rank.to_csv(OUT / "cirrus_bel_day_ranking.csv", index=False)
print(f"pixels above 25 mm in 1 h on at least 5% of the days (fixed points, not rain): {int(persistent.sum())}")
print(rank.sort_values("area_km2", ascending=False).head(30).to_string(index=False))

## Check against the reported events

The events of `docs/method.md`, *Reported events* (dates as in the reports, local time; a night event is listed
under both days). The rank of each day among all days of 7/2024–11/2026.

In [ ]:
REPORTED = {
    1: ["20240709"], 2: ["20240712"], 3: ["20240731", "20240801"], 4: ["20240802"], 5: ["20240813", "20240814"],
    6: ["20240820"], 7: ["20240902"], 8: ["20240905", "20240906"], 9: ["20240925", "20240926"], 10: ["20241009", "20241010"],
    11: ["20250108", "20250109"], 12: ["20250124", "20250125", "20250126"], 13: ["20250531"], 14: ["20250607"],
    15: ["20250613", "20250614", "20250615"], 16: ["20250625", "20250626"], 17: ["20250702"], 18: ["20250706", "20250707"],
    19: ["20250723"], 20: ["20250724"], 21: ["20250727"], 22: ["20250829"], 23: ["20250903"], 24: ["20250908", "20250909"],
    25: ["20250913"], 26: ["20260108", "20260109", "20260110"], 27: ["20260502"], 28: ["20260504"], 29: ["20260505", "20260506"],
    30: ["20260529", "20260530", "20260531"], 31: ["20260619"], 32: ["20260620", "20260621"], 33: ["20260627", "20260628"],
    34: ["20260716"], 35: ["20260819"], 36: ["20260827", "20260828"], 37: ["20260913"], 38: ["20261022", "20261023", "20261024", "20261025"],
}
by_day = rank.set_index("day")
out = []
for n, days in REPORTED.items():
    have = [d for d in days if d in by_day.index]
    best = min((by_day.loc[d, "rank"] for d in have), default=None)
    out.append((n, "/".join(d[6:] + "." + d[4:6] for d in days) + "." + days[0][:4], best,
                max((by_day.loc[d, "max_1h_mm"] for d in have), default=np.nan)))
chk = pd.DataFrame(out, columns=["event", "days", "best_rank", "max_1h_mm"])
print(chk.to_string(index=False))
print(f"events with a day in the top 40 of {len(by_day)} days: {int((chk.best_rank <= 40).sum())} of {int(chk.best_rank.notna().sum())} with CIRRUS")
top = rank.sort_values("area_km2", ascending=False).head(40)
listed = {d for days in REPORTED.values() for d in days}
print("top-40 days not in the reports:", ", ".join(d for d in top.day if d not in listed))

## Maps of the highest-ranked days

In [ ]:
top8 = rank.sort_values("area_km2", ascending=False).head(8).day.tolist()
fig, ax = plt.subplots(2, 4, figsize=(22, 9), layout="constrained")
for a_, d in zip(ax.ravel(), top8):
    im = a_.imshow(np.where(BEL, H1[DAYS.index(d)], np.nan), vmin=0, vmax=60, cmap="turbo")
    a_.set_title(f"{d[6:]}/{d[4:6]}/{d[:4]}: highest 1 h sum per pixel", fontsize=10)
    a_.set_xticks([]); a_.set_yticks([])
fig.colorbar(im, ax=ax, shrink=0.7, label="mm (index)")
plt.show()

## What is seen

Run on 10/11/2026, 848 days (1/7/2024–9/11/2026).

- **The top of the ranking is the reported convective events:** 1/8/2024 (Leuven, Tienen, Tournai) first,
  20/6/2026 (Roeselare) second, 30/5/2026 (Charleroi, Basse-Sambre) third, 9/7/2024 fourth, 27–28/6/2026 fifth.
  21 of the 37 reported events covered by CIRRUS have a day in the top 40.
- **The large-scale frontal events rank low,** as expected from a convective criterion: 8–9/1/2025 131st,
  24–26/1/2025 100th, 8–10/1/2026 329th, 22–25/10/2026 117th.
- **Events of wind, hail or very local rain rank in the middle or low:** 7/6/2025 (tornado) 49th, 3/9/2025 46th,
  23/7/2025 (Sambreville, 83 mm, very local) 56th, 16/7/2026 173rd, 13/9/2026 497th.
- **12 days of the top 40 have no report**, among them 18/6/2026, 7/9/2024, 3/8/2026 and 31/7/2025: candidates to
  check.
- **The highest 1 h sum is about 99.9 mm on most days:** the 55 dBZ cap reached by fixed points (1,236 pixels
  are above 25 mm in 1 h on at least 5% of the days). The object area, not the maximum, is the usable key until
  the H2 cleaning is applied.